In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys 
sys.path.append('../src')

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
!pwd

/home/dtc/work/project/RESCUE/notebooks


In [10]:
import torch
import shutil
import os
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from huggingface_hub import snapshot_download
from rescue.utils import sample_video
from rescue import ges_utils
from mapanything.utils.hf_utils.viz import predictions_to_glb, image_mesh, triangulate, remove_unreferenced_vertices, integrate_camera_into_scene, apply_scene_alignment 
from mapanything.utils.geometry import depthmap_to_world_frame
import numpy as np
import matplotlib
import trimesh

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
save_path = snapshot_download("facebook/map-anything", repo_type="model", local_dir="../../generated/map-anything")
model = MapAnything.from_pretrained(save_path).to(device)

In [4]:
temp_dir_path = '../../generated/temp'
if os.path.isdir(temp_dir_path):
    shutil.rmtree(temp_dir_path)

os.makedirs(temp_dir_path, exist_ok=True)
files_dir, sampled_indices = sample_video('../../generated/input/disaster_city_lawnmow.mp4', fps=1, output_dir =temp_dir_path)

In [5]:
num_views = 100

# views = load_images('../generated/sampled_disaster_city/')[:num_views]
views = load_images(files_dir)[:num_views]

poses, c2w_list, K_list, orig_width, orig_height = ges_utils.convert_ges_to_mapanything_from_file('../../generated/input/disaster_city_lawnmow.json', ref_frame = 0)
poses = [poses[i] for i in sampled_indices[:num_views]]
c2w_list = [c2w_list[i] for i in sampled_indices[:num_views]]
K_list = [K_list[i] for i in sampled_indices[:num_views]]

assert len(views) == len(poses) == len(c2w_list) == len(K_list), 'Lengths of views, poses, c2w_list, and K_list must be equal'

for i in range(len(views)):
    h_new, w_new = views[i]['true_shape'][0]   
    K = K_list[i].copy()
    scale_x = w_new / orig_width            
    scale_y = h_new / orig_height           
    K[0, 0] *= scale_x   # fl_x
    K[0, 2] *= scale_x   # cx
    K[1, 1] *= scale_y   # fl_y
    K[1, 2] *= scale_y   # cy

    device = views[i]['img'].device
    views[i]['intrinsics']    = torch.tensor(K, dtype=torch.float32, device=device)[None]
    views[i]['camera_poses']  = torch.tensor(c2w_list[i], dtype=torch.float32, device=device)[None]
    views[i]['is_metric_scale'] = False

In [6]:
import numpy as np

# Take frame 0 - its ENU position should be exactly [0,0,0]
# and its forward vector (col 2 of R) should point roughly downward (nadir view)
c2w_0 = c2w_list[0]   # before gl2cv
print("Position (should be ~[0,0,0]):", c2w_0[:3, 3])

# Camera axes in ENU world
right   = c2w_0[:3, 0]   # should point East-ish
up_cam  = c2w_0[:3, 1]   # in OpenCV: should point DOWN in world
forward = c2w_0[:3, 2]   # in OpenCV: should point INTO scene (roughly toward ground)

print("Forward vector (col 2):", np.round(forward, 3))
print("  Z component:", forward[2], "← negative means pointing downward (correct for nadir)")


Position (should be ~[0,0,0]): [0. 0. 0.]
Forward vector (col 2): [ 0. -0. -1.]
  Z component: -1.0 ← negative means pointing downward (correct for nadir)


In [8]:
# views[0].keys()

In [9]:
# # Verify: view i's idx field should match i
# assert all(views[i]['idx'] == i for i in range(len(views))), "load_images reordered files"

# # Verify: sampled_indices should be strictly increasing (no frame reordering)
# assert all(sampled_indices[i] < sampled_indices[i+1] for i in range(len(sampled_indices)-1)), \
#     "sampled_indices not monotonically increasing"

# print(f"First 5 sampled source frames: {sampled_indices[:5]}")
# print(f"views[0] true_shape: {views[0]['true_shape']}")

In [8]:
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=True,  # Trades off speed for more views (up to 2000 views on 140 GB). Trade off is negligible - see profiling section
    minibatch_size=None,              # Minibatch size for memory-efficient inference (use 1 for smallest GPU memory consumption). Default is dynamic computation based on available GPU memory.
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    use_multiview_confidence=True,   # Enable multi-view depth consistency based confidence in place of learning-based one
)

In [ ]:

# Full object graph (list of per-view dicts with tensors) — use torch.save, not pickle of raw bytes alone
predictions_path = os.path.join("../../generated/output", "predictions.pt")
os.makedirs(os.path.dirname(predictions_path), exist_ok=True)
torch.save(predictions, predictions_path)
print(f"Saved predictions ({len(predictions)} views) to {predictions_path}")

In [7]:

# Reload saved inference output (skip model.infer) and continue with the GLB cell below.
predictions_path = os.path.join("../../generated/output", "predictions.pt")
_load_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictions = torch.load(
    predictions_path,
    map_location=_load_device,
    weights_only=False,  # required for lists/dicts of tensors (PyTorch 2.6+)
)
print(f"Loaded predictions ({len(predictions)} views) from {predictions_path}")

Loaded predictions (67 views) from ../../generated/output/predictions.pt


In [13]:
def align_and_deduplicate_meshes(
    source_mesh: trimesh.Trimesh,
    target_mesh: trimesh.Trimesh,
    threshold: float = 1e-5,
    max_iterations: int = 50,
    merge_epsilon: float = 1e-5,
) -> trimesh.Trimesh:
    """
    Aligns a target mesh to a source mesh with ICP and removes duplicate vertices.

    Args:
        source_mesh (trimesh.Trimesh): Reference mesh.
        target_mesh (trimesh.Trimesh): Mesh to be aligned and deduplicated.
        threshold (float): Convergence threshold for ICP.
        max_iterations (int): Maximum ICP iterations.
        merge_epsilon (float): Maximum distance for vertex merging.

    Returns:
        trimesh.Trimesh: A new aligned mesh with duplicate vertices removed.
    """
    if not isinstance(source_mesh, trimesh.Trimesh):
        raise ValueError("source_mesh must be a trimesh.Trimesh")
    if not isinstance(target_mesh, trimesh.Trimesh):
        raise ValueError("target_mesh must be a trimesh.Trimesh")

    source_points = source_mesh.vertices.copy()
    target_points = target_mesh.vertices.copy()

    result = trimesh.registration.icp(
        source_points,
        target_points,
        threshold=threshold,
        max_iterations=max_iterations,
    )

    transform = result[0] if isinstance(result, tuple) else result
    alignment_mesh = target_mesh.copy()
    aliged_mesh = target_mesh.copy()
    alignment_mesh.apply_transform(transform)

    if merge_epsilon > 0:
        digits = max(0, int(np.ceil(-np.log10(merge_epsilon))))
        alignment_mesh.merge_vertices(digits_vertex=digits)
    aliged_mesh.update_faces(alignment_mesh.unique_faces())
    aliged_mesh.remove_unreferenced_vertices()

    return aliged_mesh


def predictions_to_glb_alligned(
    predictions,
    filter_by_frames="all",
    mask_black_bg=False,
    mask_white_bg=False,
    show_cam=True,
    mask_ambiguous=False,
    as_mesh=True,
    conf_percentile=None,
) -> trimesh.Scene:
    """
    Converts MapAnything predictions to a 3D scene represented as a GLB file.

    Args:
        predictions (dict): Dictionary containing model predictions with keys:
            - world_points: 3D point coordinates (S, H, W, 3)
            - images: Input images (S, H, W, 3)
            - extrinsic: Camera extrinsic matrices (S, 3, 4)
        filter_by_frames (str): Frame filter specification (default: "all")
        mask_black_bg (bool): Mask out black background pixels (default: False)
        mask_white_bg (bool): Mask out white background pixels (default: False)
        show_cam (bool): Include camera visualization (default: True)
        mask_ambiguous (bool): Apply final mask to filter ambiguous predictions (default: False)
        as_mesh (bool): Represent the data as a mesh instead of point cloud (default: False)

    Returns:
        trimesh.Scene: Processed 3D scene containing point cloud/mesh and cameras

    Raises:
        ValueError: If input predictions structure is invalid
    """
    if not isinstance(predictions, dict):
        raise ValueError("predictions must be a dictionary")

    print("Building GLB scene")
    selected_frame_idx = None
    if filter_by_frames != "all" and filter_by_frames != "All":
        try:
            # Extract the index part before the colon
            selected_frame_idx = int(filter_by_frames.split(":")[0])
        except (ValueError, IndexError):
            pass

    # Always use Pointmap Branch
    print("Using Pointmap Branch")
    if "world_points" not in predictions:
        raise ValueError(
            "world_points not found in predictions. Pointmap Branch requires 'world_points' key. "
            "Depthmap and Camera branches have been removed."
        )

    pred_world_points = predictions["world_points"]

    # Get images from predictions
    images = predictions["images"]
    # Use extrinsic matrices instead of pred_extrinsic_list
    camera_matrices = predictions["extrinsic"]

    if selected_frame_idx is not None:
        pred_world_points = pred_world_points[selected_frame_idx][None]
        images = images[selected_frame_idx][None]
        camera_matrices = camera_matrices[selected_frame_idx][None]

    vertices_3d = pred_world_points.reshape(-1, 3)
    # Handle different image formats - check if images need transposing
    if images.ndim == 4 and images.shape[1] == 3:  # NCHW format
        colors_rgb = np.transpose(images, (0, 2, 3, 1))
    else:  # Assume already in NHWC format
        colors_rgb = images
    colors_rgb = (colors_rgb.reshape(-1, 3) * 255).astype(np.uint8)

    # Create mask for filtering
    mask = np.ones(len(vertices_3d), dtype=bool)
    final_mask = predictions["final_mask"].reshape(-1)

    # Confidence masking
    if conf_percentile is not None and "conf" in predictions:
        # print ("Applying confidence masking...")
        conf = predictions["conf"].reshape(-1)
        threshold = np.percentile(conf, conf_percentile)
        # print (f"Confidence threshold at {conf_percentile} percentile: {threshold}")
        conf_mask = conf >= threshold
        mask = mask & conf_mask

    if mask_black_bg:
        black_bg_mask = colors_rgb.sum(axis=1) >= 16
        mask = mask & black_bg_mask

    if mask_white_bg:
        # Filter out white background pixels (RGB values close to white)
        # Consider pixels white if all RGB values are above 240
        white_bg_mask = (
            (colors_rgb[:, 0] > 240)
            & (colors_rgb[:, 1] > 240)
            & (colors_rgb[:, 2] > 240)
        )
        mask = mask & ~white_bg_mask

    # Use final_mask when mask_ambiguous is checked
    if mask_ambiguous:
        mask = mask & final_mask

    vertices_3d = vertices_3d[mask].copy()
    colors_rgb = colors_rgb[mask].copy()

    if vertices_3d is None or np.asarray(vertices_3d).size == 0:
        vertices_3d = np.array([[1, 0, 0]])
        colors_rgb = np.array([[255, 255, 255]])
        scene_scale = 1
    else:
        # Calculate the 5th and 95th percentiles along each axis
        lower_percentile = np.percentile(vertices_3d, 5, axis=0)
        upper_percentile = np.percentile(vertices_3d, 95, axis=0)

        # Calculate the diagonal length of the percentile bounding box
        scene_scale = np.linalg.norm(upper_percentile - lower_percentile)

    colormap = matplotlib.colormaps.get_cmap("gist_rainbow")

    # Initialize a 3D scene
    scene_3d = trimesh.Scene()

    # Add point cloud data to the scene
    if as_mesh:
        # Create mesh from pointcloud
        # try:
        if selected_frame_idx is not None:
            # Single frame case - we can create a proper mesh
            H, W = pred_world_points.shape[1:3]

            # Get original unfiltered data for mesh creation
            original_points = pred_world_points.reshape(H, W, 3)

            # Reshape original image data properly
            if images.ndim == 4 and images.shape[1] == 3:  # NCHW format
                original_image_colors = np.transpose(images[0], (1, 2, 0))
            else:  # Assume already in HWC format
                original_image_colors = images[0]
            original_image_colors *= 255
            # Get original final mask
            original_final_mask = predictions["final_mask"][selected_frame_idx].reshape(
                H, W
            )

            # Create mask based on final mask
            mask = original_final_mask

            # Confidence masking
            if conf_percentile is not None and "conf" in predictions:
                # print ("Applying confidence masking...")
                conf = predictions["conf"][selected_frame_idx].reshape(-1)
                threshold = np.percentile(conf, conf_percentile)
                # print (f"Confidence threshold at {conf_percentile} percentile: {threshold}")
                conf_mask = conf >= threshold
                mask = mask & conf_mask.reshape(H, W)

            # Additional background masks if needed
            if mask_black_bg:
                black_bg_mask = original_image_colors.sum(axis=2) >= 16
                mask = mask & black_bg_mask

            if mask_white_bg:
                white_bg_mask = ~(
                    (original_image_colors[:, :, 0] > 240)
                    & (original_image_colors[:, :, 1] > 240)
                    & (original_image_colors[:, :, 2] > 240)
                )
                mask = mask & white_bg_mask

            # Check if normals are available in predictions
            vertex_normals = None
            if "normal" in predictions and predictions["normal"] is not None:
                # Get normals for the selected frame
                frame_normals = (
                    predictions["normal"][selected_frame_idx]
                    if selected_frame_idx is not None
                    else predictions["normal"][0]
                )

                # Create faces and vertices using image_mesh with normals support
                faces, vertices, vertex_colors, vertex_normals = image_mesh(
                    original_points * np.array([1, -1, 1], dtype=np.float32),
                    original_image_colors / 255.0,
                    frame_normals * np.array([1, -1, 1], dtype=np.float32),
                    mask=mask,
                    tri=True,
                    return_indices=False,
                )

                # Apply coordinate transformations to normals
                vertex_normals = vertex_normals * np.array([1, -1, 1], dtype=np.float32)
            else:
                # Create faces and vertices using image_mesh without normals
                faces, vertices, vertex_colors = image_mesh(
                    original_points * np.array([1, -1, 1], dtype=np.float32),
                    original_image_colors / 255.0,
                    mask=mask,
                    tri=True,
                    return_indices=False,
                )

            # vertices = vertices * np.array([1, -1, 1], dtype=np.float32)

            # Create trimesh object with optional normals
            mesh_data = trimesh.Trimesh(
                vertices=vertices * np.array([1, -1, 1], dtype=np.float32),
                faces=faces,
                vertex_colors=(vertex_colors * 255).astype(np.uint8),
                vertex_normals=(vertex_normals if vertex_normals is not None else None),
                process=False,
            )
            scene_3d.add_geometry(mesh_data)

        else:
            # Multi-frame case - create separate meshes for each frame
            print("Creating mesh for multi-frame data...")
            prev_frame_points=None

            for frame_idx in range(pred_world_points.shape[0]):
                H, W = pred_world_points.shape[1:3]

                # Get data for this frame
                frame_points = pred_world_points[frame_idx]
                frame_final_mask = predictions["final_mask"][frame_idx]

                # Get frame image
                if images.ndim == 4 and images.shape[1] == 3:  # NCHW format
                    frame_image = np.transpose(images[frame_idx], (1, 2, 0))
                else:  # Assume already in HWC format
                    frame_image = images[frame_idx]
                frame_image *= 255
                # Create mask for this frame using final_mask
                mask = frame_final_mask

                # Additional background masks if needed
                if mask_black_bg:
                    black_bg_mask = frame_image.sum(axis=2) >= 16
                    mask = mask & black_bg_mask

                if mask_white_bg:
                    white_bg_mask = ~(
                        (frame_image[:, :, 0] > 240)
                        & (frame_image[:, :, 1] > 240)
                        & (frame_image[:, :, 2] > 240)
                    )
                    mask = mask & white_bg_mask
                if conf_percentile is not None and "conf" in predictions:
                    # print ("Applying confidence masking...")
                    conf = predictions["conf"][frame_idx].reshape(-1)
                    threshold = np.percentile(conf, conf_percentile)
                    # print (f"Confidence threshold at {conf_percentile} percentile: {threshold}")
                    conf_mask = conf >= threshold
                    mask = mask & conf_mask.reshape(H, W)
                # Create mesh for this frame
                faces, vertices, vertex_colors = image_mesh(
                    frame_points * np.array([1, -1, 1], dtype=np.float32),
                    frame_image / 255.0,
                    mask=mask,
                    tri=True,
                    return_indices=False,
                )

                vertices = vertices * np.array([1, -1, 1], dtype=np.float32)
                # Create trimesh object for this frame
                frame_mesh = trimesh.Trimesh(
                    vertices=vertices,
                    faces=faces,
                    vertex_colors=(vertex_colors * 255).astype(np.uint8),
                    process=False,
                )
                if prev_frame_points is not None:
                    aligned_mesh = align_and_deduplicate_meshes(
                        prev_frame_points,
                        frame_mesh,
                        #threshold=scene_scale * 0.01,
                        #max_iterations=100,
                        #merge_epsilon=scene_scale * 0.005,
                    )
                    scene_3d.add_geometry(aligned_mesh)
                    prev_frame_points = frame_mesh #scene_3d.to_mesh()
                else:
                    scene_3d.add_geometry(frame_mesh)
                    prev_frame_points = frame_mesh #scene_3d.to_mesh()
    else:
        point_cloud_data = trimesh.PointCloud(vertices=vertices_3d, colors=colors_rgb)
        scene_3d.add_geometry(point_cloud_data)

    # Prepare 4x4 matrices for camera extrinsics
    num_cameras = len(camera_matrices)

    if show_cam:
        # Add camera models to the scene
        for i in range(num_cameras):
            world_to_camera = camera_matrices[i]
            rgba_color = colormap(i / num_cameras)
            current_color = tuple(int(255 * x) for x in rgba_color[:3])

            integrate_camera_into_scene(
                scene_3d, world_to_camera, current_color, scene_scale
            )

    # Align scene to the observation of the first camera
    scene_3d = apply_scene_alignment(scene_3d, camera_matrices)

    print("GLB Scene built")
    return scene_3d


In [ ]:
# Build the predictions dict that predictions_to_glb expects
world_points_list = []
images_list       = []
extrinsic_list    = []
mask_list         = []

for pred in predictions:
    pts3d, valid_mask = depthmap_to_world_frame(
        pred["depth_z"][0].squeeze(-1),
        pred["intrinsics"][0],
        pred["camera_poses"][0],
    )

    mask = pred["mask"][0].squeeze(-1).cpu().numpy().astype(bool)
    mask = mask & valid_mask.cpu().numpy()

    world_points_list.append(pts3d.cpu().numpy())
    images_list.append(pred["img_no_norm"][0].cpu().numpy())
    extrinsic_list.append(pred["camera_poses"][0].cpu().numpy())   # (4, 4) cam2world
    mask_list.append(mask)

pred_dict = {
    "world_points": np.stack(world_points_list),  # (S, H, W, 3)
    "images":       np.stack(images_list),         # (S, H, W, 3)
    "extrinsic":    np.stack(extrinsic_list),      # (S, 3, 4)
    "final_mask":   np.stack(mask_list),           # (S, H, W)
}
print("came here")
# Export to GLB
glbscene = predictions_to_glb_alligned(
    pred_dict,
    show_cam=True,
    conf_percentile=0,   # 0 = keep all points
)
glbscene.export("../../generated/output/reconstruction4.glb")

# Additional step: load the exported GLB, convert/copy it to Blender specs, and save as a separate GLB for Blender



came here
Building GLB scene
Using Pointmap Branch
Creating mesh for multi-frame data...


In [1]:
# Convert the exported GLB to Blender's coordinate system (Z-up).
import trimesh
import numpy as np

input_glb_path = "../../generated/output/reconstruction.glb"
output_blender_glb_path = "../../generated/output/reconstruction_blender.glb"

# Load the mesh or scene
mesh_or_scene = trimesh.load(input_glb_path)

# Rotate -90 degrees (not +90!) around X-axis to convert Y-up (GLB default) to Z-up (Blender)
# A positive 90 deg will flip Z-down instead of Z-up (Blender expects Z-up as positive)
transform = trimesh.transformations.rotation_matrix(np.radians(-90), [1, 0, 0])

if isinstance(mesh_or_scene, trimesh.Scene):
    # Apply transform to all geometries in the scene
    for geom in mesh_or_scene.geometry.values():
        geom.apply_transform(transform)
    mesh_or_scene.export(output_blender_glb_path)
else:
    mesh_or_scene.apply_transform(transform)
    mesh_or_scene.export(output_blender_glb_path)

print(f"Saved Z-up GLB (for Blender) to {output_blender_glb_path}")